# 03 Stationarity and Cointegration

> **Notebook status**: This is a standalone teaching notebook.  The production
> pipeline (`python -m src.model`) and diagnostic module
> (`python -m src.diagnostics`) produce the same ADF / cointegration results
> with additional export formatting.  Note that this notebook uses
> `autolag="AIC"`; the Stata scripts use fixed `lags(1)`.

ADF tests and Engle-Granger residual-based cointegration test.

In [ ]:
import pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller
from pathlib import Path
ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
df = pd.read_csv(ROOT / "data/processed/moutai_profit_processed.csv")

In [ ]:
def adf_result(series):
    stat, p, lags, nobs, crit, icbest = adfuller(series.dropna(), autolag="AIC")
    return {"ADF statistic": stat, "p-value": p, "lags": lags, "5% critical": crit["5%"]}

for col in ["LNY","LNX2","LNX3","LNX4","LNX5","DLNY","DLNX2","DLNX3","DLNX4","DLNX5"]:
    print(col, adf_result(df[col]))

In [ ]:
y = df["LNY"]
x = sm.add_constant(df[["LNX2","LNX3","LNX4","LNX5"]])
long_run = sm.OLS(y, x, missing="drop").fit()
df["ECM"] = long_run.resid
print(long_run.summary())
print(adf_result(df["ECM"]))